In [45]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import json
import gseapy as gp

In [115]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

In [116]:
comm_idx = int(input("Community Index: "))

# Loading

In [117]:
def load(d):
    with open(f"../output/{d}/result_communities_selected.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"../output/{d}/result_communities_HGNC_selected.pkl", "rb") as f:
        communities_HGNC = pickle.load(f)
    # with open(f"../output/{d}/leiden_results/result_graph.pkl", "rb") as f:
    #     graph = pickle.load(f)    
    with open(f"../output/{d}/gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
        
    return communities,communities_HGNC,gene_to_index_distinct

In [118]:
communities_selected,communities_HGNC_selected,gene_to_index_distinct = load(DISEASE)

# index to HGNC

In [119]:
index_to_gene_distinct = {v: u for (u,v) in gene_to_index_distinct.items()}

In [120]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

# Community deepdive

In [121]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [122]:
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value),fisher_z
0,1,1031,RNA Binding (GO:0003723),359/1411,2.531771e-160,binding,GO_Molecular_Function_2023,6.160028e-163,0.0,0.0,9.098603,3398.358251,POP5;SLC4A1AP;POP1;RTCA;RRP1;PPAN;FCF1;HNRNPU;...,0.254429,0.800313
1,1,1031,Metabolism Of RNA R-HSA-8953854,198/666,9.105476e-96,Metabolism of RNA,Reactome_2022,2.474314e-98,0.0,0.0,9.396574,2111.855352,LTV1;POP5;FCF1;HNRNPU;HNRNPR;PHAX;PWP2;RRP9;CC...,0.297297,1.577535
2,1,1031,mRNA Processing (GO:0006397),100/214,1.048400e-67,cellular process,GO_Biological_Process_2023,5.939942e-71,0.0,0.0,17.765278,2872.678156,DBR1;GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRP...,0.467290,1.577535
3,1,1031,"RNA Splicing, Via Transesterification Reaction...",92/180,7.059510e-67,cellular process,GO_Biological_Process_2023,7.999445e-70,0.0,0.0,21.021541,3344.560525,DBR1;HNRNPU;PPWD1;HNRNPR;CASC3;CWC27;PNN;SYNCR...,0.511111,1.577535
4,1,1031,"mRNA Splicing, Via Spliceosome (GO:0000398)",98/211,1.931445e-66,cellular process,GO_Biological_Process_2023,3.282909e-69,0.0,0.0,17.527322,2763.877077,DBR1;GEMIN2;HNRNPU;HNRNPR;CASC3;CWC27;PNN;SNRP...,0.464455,1.577535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1462,11,118,G Protein-Coupled Acetylcholine Receptor Signa...,6/17,4.955206e-08,cellular process,GO_Biological_Process_2023,4.353615e-10,0.0,0.0,96.774351,2085.956059,CHRM2;CHRM3;CHRM1;CHRM4;CHRM5;HRH4,0.352941,2.155359
1463,11,118,Adrenergic Receptor Signaling Pathway (GO:0071...,6/18,6.780770e-08,cellular process,GO_Biological_Process_2023,6.499140e-10,0.0,0.0,88.705357,1876.489185,ADRB3;ADRB2;ADRA2C;ADRA2B;ADRA2A;DRD5,0.333333,2.155359
1464,11,118,Xenobiotic Catabolic Process (GO:0042178),6/19,9.104195e-08,cellular process,GO_Biological_Process_2023,9.453237e-10,0.0,0.0,81.877747,1701.378131,GSTM3;CYP2C8;GSTM1;CYP2B6;TPMT;CYP3A5,0.315789,2.155359
1465,11,118,Graft-versus-host disease,6/42,4.755501e-06,Immune disease,KEGG_2021_Human,1.636587e-07,0.0,0.0,29.532738,461.463292,IL1A;HLA-DRB5;HLA-DPB1;HLA-A;HLA-DRB3;IL2,0.142857,-0.939496


In [123]:
go_df_filtered = important_terms[important_terms["Community Index"] == comm_idx]
go_df_filtered = go_df_filtered.sort_values(by = ["fisher_z"], ascending = [False])

In [124]:
# sort by customized formula: -log10(p-value) * (overlap / set_size)
go_df_filtered["custom_score"] = -np.log10(go_df_filtered["Adjusted P-value"]) + np.log(1+len(go_df_filtered["Genes"]))

In [125]:
go_df_filtered = go_df_filtered.sort_values(by = ["fisher_z","custom_score"], ascending = [False,False])

In [130]:
# Show only a few columns
go_df_filtered[["Community Index", "Term", "Category",  "Adjusted P-value", "Overlap", "fisher_z", "custom_score"]]

,Community Index,Term,Category,Adjusted P-value,Overlap,fisher_z,custom_score
301,4,Regulation Of DNA-templated Transcription (GO:...,biological regulation,1.584710e-38,230/1922,2.062651,42.675247
302,4,Regulation Of Transcription By RNA Polymerase ...,biological regulation,4.700991e-33,226/2028,2.062651,37.203008
304,4,Negative Regulation Of DNA-templated Transcrip...,biological regulation,6.992709e-29,142/1025,2.062651,33.030552
307,4,Negative Regulation Of Transcription By RNA Po...,biological regulation,1.624892e-24,112/763,2.062651,28.664373
320,4,Positive Regulation Of DNA-templated Transcrip...,biological regulation,3.058956e-14,127/1243,2.062651,18.389624
...,...,...,...,...,...,...,...
312,4,Regulation Of Nucleic Acid-Templated Transcrip...,NaN,2.219218e-18,74/452,NaN,22.528997
326,4,Negative Regulation Of Nucleic Acid-Templated ...,NaN,1.258144e-12,64/456,NaN,16.775467
357,4,Positive Regulation Of Nucleic Acid-Templated ...,NaN,1.681958e-08,63/557,NaN,12.649382
359,4,Nuclear Receptor Coactivator Activity (GO:0030...,NaN,2.382211e-08,16/50,NaN,12.498217
